In [ ]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

# =========================
# (1) 입력 경로 설정
# =========================
XLSX_PATH = r"/home/a202192020/맥주데이터실험/data/Supplemental Files and Figure source files.xlsx"

# ✅ 실험 결과(압축 해제된 run 폴더) 경로를 넣어줘
EXP_RUN_DIR = r"/home/a202192020/맥주데이터실험/pca_add/0225/output/20260226_014121"

# 비교 결과 저장 위치
OUT_DIR = Path(EXP_RUN_DIR) / "종합"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / f"compare_paper_vs_experiment_best_test_r2_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"


# =========================
# (2) 논문(xlsx)에서 관능 50개별 "best R2" 뽑기
# =========================
# 논문 성능표: Table S3 (50행, 여러 모델 컬럼)
paper_tbl = pd.read_excel(XLSX_PATH, sheet_name="Supplementary Table S3")
# target 코드 ↔ 논문 aspect 이름 매핑: Sensory names (Trained panel 50개)
names = pd.read_excel(XLSX_PATH, sheet_name="Sensory names")
map_tp = names[names["dataset"].str.lower().eq("trained panel")].copy()

# Table S3의 aspect 이름은 Sensory names의 CleanName_PanelFeedback과 1:1로 일치함(50개)
# model 컬럼들(= R2 값들)
model_cols = [c for c in paper_tbl.columns if c != "aspect"]

paper_tbl["paper_best_r2"] = paper_tbl[model_cols].max(axis=1)
paper_tbl["paper_best_model"] = paper_tbl[model_cols].idxmax(axis=1)

# aspect 기준으로 조인해서, target 코드(R_name) 단위로 논문 best R2 얻기
paper_best = (
    map_tp[["R_name", "CleanName_PanelFeedback"]]
    .merge(paper_tbl[["aspect", "paper_best_r2", "paper_best_model"]],
           left_on="CleanName_PanelFeedback", right_on="aspect", how="left")
    .rename(columns={"R_name": "target_code", "CleanName_PanelFeedback": "aspect_name"})
    .drop(columns=["aspect"])
)

# sanity
missing = paper_best["paper_best_r2"].isna().sum()
print("paper_best missing:", missing, "/ 50")


# =========================
# (3) 실험 결과 폴더에서 target별 best test R2 뽑기
# =========================
exp_dir = Path(EXP_RUN_DIR)

# 우선순위 1) cumulative_pcr_all_targets.csv (k별 누적 PCR 로그)
cand1 = list(exp_dir.rglob("cumulative_pcr_all_targets.csv"))
# 우선순위 2) single_pc_results_all_targets.csv (PC 1개 실험)
cand2 = list(exp_dir.rglob("single_pc_results_all_targets.csv"))

if cand1:
    df = pd.read_csv(cand1[0])
    df["test_r2"] = pd.to_numeric(df["test_r2"], errors="coerce")
    df["k"] = pd.to_numeric(df["k"], errors="coerce")
    df = df.dropna(subset=["target", "test_r2", "k"])

    idx = df.groupby("target")["test_r2"].idxmax()
    exp_best = df.loc[idx, ["target", "test_r2", "k"]].copy()
    exp_best = exp_best.rename(columns={"target": "target_code", "test_r2": "exp_best_test_r2", "k": "exp_best_arg"})
    exp_best["exp_best_arg"] = exp_best["exp_best_arg"].astype(int)
    exp_best["exp_source"] = str(cand1[0])

elif cand2:
    df = pd.read_csv(cand2[0])
    df["test_r2"] = pd.to_numeric(df["test_r2"], errors="coerce")
    df = df.dropna(subset=["target", "test_r2"])

    # pc index 컬럼 탐색
    pc_col = "pc_index_1based_full" if "pc_index_1based_full" in df.columns else (
        "pc_index_1based_kept" if "pc_index_1based_kept" in df.columns else None
    )

    idx = df.groupby("target")["test_r2"].idxmax()
    cols = ["target", "test_r2"] + ([pc_col] if pc_col else [])
    exp_best = df.loc[idx, cols].copy()
    exp_best = exp_best.rename(columns={"target": "target_code", "test_r2": "exp_best_test_r2"})
    if pc_col:
        exp_best = exp_best.rename(columns={pc_col: "exp_best_arg"})
    else:
        exp_best["exp_best_arg"] = np.nan
    exp_best["exp_source"] = str(cand2[0])

else:
    raise FileNotFoundError("실험 결과 폴더에서 cumulative_pcr_all_targets.csv 또는 single_pc_results_all_targets.csv를 찾지 못했습니다.")


# =========================
# (4) 비교 테이블 생성 + 저장
# =========================
compare = paper_best.merge(exp_best, on="target_code", how="outer")
compare["delta(exp - paper)"] = compare["exp_best_test_r2"] - compare["paper_best_r2"]

# 보기 좋게 정렬: 실험이 논문보다 좋아진 순
compare = compare.sort_values("delta(exp - paper)", ascending=False)

compare.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("✅ saved:", OUT_CSV)

# 상위 15개 출력
display(compare.head(50)[[
    "target_code","aspect_name",
    "paper_best_r2","paper_best_model",
    "exp_best_test_r2","exp_best_arg",
    "delta(exp - paper)"
]])

paper_best missing: 0 / 50
✅ saved: /home/a202192020/맥주데이터실험/pca_add/0225/output/20260226_014121/종합/compare_paper_vs_experiment_best_test_r2_20260227_171542.csv


,target_code,aspect_name,paper_best_r2,paper_best_model,exp_best_test_r2,exp_best_arg,delta(exp - paper)
2,A_esters_flower,Esters aroma - floral,0.03,AdaBoost,0.308222,44,0.278222
17,F_esters_flower,Esters taste - floral,0.00,AdaBoost,0.084582,5,0.084582
12,A_malt_burn,Malt aroma - smoked,0.23,AdaBoost,0.295572,66,0.065572
6,A_hops_citrus,Hops aroma - citrus,0.05,ExtraTrees,0.108728,2,0.058728
10,A_malt_all,Malt aroma - overall,0.41,AdaBoost,0.467233,26,0.057233
45,orange,Orange,0.27,GradientBoost,0.324242,13,0.054242
25,F_malt_all,Malt taste - overall,0.42,AdaBoost,0.461371,32,0.041371
37,body,Body,0.55,AdaBoost,0.579284,25,0.029284
18,F_esters_fruity,Esters taste - fruity,0.18,XGBoost,0.195827,9,0.015827
19,F_esters_isoaa,Esters taste - banana,0.18,AdaBoost,0.178217,4,-0.001783


9개 관능에 대한 R^2만 좋아지고 나머지는 좋아지지 않음

## 두 결과 비교 csv 저장

In [ ]:
import os
import re
import zipfile
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd


# =========================
# 0) 여기만 네 경로로 바꿔줘
# =========================
PAPER_RESULT = r"/mnt/data/20260225_220244.zip"     # 기존 논문 방식(재현) 결과 zip 또는 폴더 경로
EXP_RESULT   = r"/mnt/data/20260226_014121.zip"     # 네 실험(PCA/PCR 등) 결과 zip 또는 폴더 경로

# 비교 결과 저장 위치(원하는 경로로 바꿔도 됨)
OUT_DIR = Path("./compare_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / f"compare_best_test_r2_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"


# =========================
# 1) 유틸: zip/dir 공통으로 파일 찾고 csv 읽기
# =========================
def _is_zip(p: str) -> bool:
    return str(p).lower().endswith(".zip")

def _find_in_zip(zip_path: str, filename: str):
    """zip 내부에서 filename으로 끝나는 첫 파일 경로 반환(없으면 None)"""
    with zipfile.ZipFile(zip_path, "r") as z:
        for n in z.namelist():
            if n.endswith(filename):
                return n
    return None

def _read_csv_zip(zip_path: str, inner_path: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(inner_path) as f:
            return pd.read_csv(f)

def _find_in_dir(dir_path: str, filename: str):
    """dir 내부에서 filename인 첫 파일 Path 반환(없으면 None)"""
    p = Path(dir_path)
    hits = list(p.rglob(filename))
    return hits[0] if hits else None

def _read_csv_dir(file_path: Path) -> pd.DataFrame:
    return pd.read_csv(file_path)

def load_csv_auto(run_path: str, filename: str):
    """
    run_path가 zip이면 zip 내부에서 filename을 찾고 읽음.
    run_path가 폴더면 하위에서 filename을 찾고 읽음.
    return: (df, where_str) or (None, None)
    """
    if _is_zip(run_path):
        inner = _find_in_zip(run_path, filename)
        if inner is None:
            return None, None
        return _read_csv_zip(run_path, inner), f"ZIP::{inner}"
    else:
        fp = _find_in_dir(run_path, filename)
        if fp is None:
            return None, None
        return _read_csv_dir(fp), f"DIR::{fp.as_posix()}"


# =========================
# 2) 핵심: run(결과)에서 target별 "best test R²" 뽑기
# =========================
def extract_best_test_r2(run_path: str, label: str) -> pd.DataFrame:
    """
    가능한 한 자동으로 결과 파일을 찾아서,
    target별 best test R²(= max) + 그때의 k/pc/model 같은 정보를 뽑아 표준화해서 반환.
    반환 컬럼(최소):
      - target
      - {label}_best_test_r2
    추가(가능할 때):
      - {label}_best_arg (k/pc/model 등)
      - {label}_source (어떤 파일에서 뽑았는지)
    """
    # 우선순위 1) 누적 PCR 결과
    cum, where = load_csv_auto(run_path, "cumulative_pcr_all_targets.csv")
    if cum is not None:
        df = cum.copy()

        # 혹시 설명행(문자열)이 섞여있으면 숫자로 강제 변환 후 제거
        df["test_r2"] = pd.to_numeric(df["test_r2"], errors="coerce")
        df["k"] = pd.to_numeric(df["k"], errors="coerce")
        df = df.dropna(subset=["target", "test_r2", "k"])

        # target별 test_r2 최대 행
        idx = df.groupby("target")["test_r2"].idxmax()
        best = df.loc[idx, ["target", "test_r2", "k"]].copy()

        best = best.rename(columns={
            "test_r2": f"{label}_best_test_r2",
            "k": f"{label}_best_arg"
        })
        best[f"{label}_best_arg"] = best[f"{label}_best_arg"].astype(int)
        best[f"{label}_source"] = where + " (groupby max test_r2 over k)"
        return best.sort_values("target").reset_index(drop=True)

    # 우선순위 2) 단일 PC 전체 결과
    sp, where = load_csv_auto(run_path, "single_pc_results_all_targets.csv")
    if sp is not None:
        df = sp.copy()
        df["test_r2"] = pd.to_numeric(df["test_r2"], errors="coerce")
        df = df.dropna(subset=["target", "test_r2"])

        # 어떤 PC가 best인지 표시할 수 있으면 같이
        pc_col = None
        for c in ["pc_index_1based_full", "pc_index_1based_kept"]:
            if c in df.columns:
                pc_col = c
                break

        idx = df.groupby("target")["test_r2"].idxmax()
        cols = ["target", "test_r2"] + ([pc_col] if pc_col else [])
        best = df.loc[idx, cols].copy()

        best = best.rename(columns={"test_r2": f"{label}_best_test_r2"})
        if pc_col:
            best = best.rename(columns={pc_col: f"{label}_best_arg"})
        else:
            best[f"{label}_best_arg"] = np.nan
        best[f"{label}_source"] = where + " (groupby max test_r2 over PCs)"
        return best.sort_values("target").reset_index(drop=True)

    # 우선순위 3) best_k_per_target (이미 요약된 경우)
    bk, where = load_csv_auto(run_path, "best_k_per_target.csv")
    if bk is not None and "best_test_r2" in bk.columns:
        df = bk.copy()
        df["best_test_r2"] = pd.to_numeric(df["best_test_r2"], errors="coerce")
        df = df.dropna(subset=["target", "best_test_r2"])

        out = df[["target", "best_test_r2"]].copy()
        out = out.rename(columns={"best_test_r2": f"{label}_best_test_r2"})
        # best_k도 있으면 best_arg로
        if "best_k" in df.columns:
            out[f"{label}_best_arg"] = pd.to_numeric(df["best_k"], errors="coerce")
        out[f"{label}_source"] = where + " (best_k_per_target)"
        return out.sort_values("target").reset_index(drop=True)

    # 우선순위 4) best_pc_per_target
    bp, where = load_csv_auto(run_path, "best_pc_per_target.csv")
    if bp is not None and "test_r2" in bp.columns:
        df = bp.copy()
        df["test_r2"] = pd.to_numeric(df["test_r2"], errors="coerce")
        df = df.dropna(subset=["target", "test_r2"])

        out = df[["target", "test_r2"]].copy()
        out = out.rename(columns={"test_r2": f"{label}_best_test_r2"})
        out[f"{label}_source"] = where + " (best_pc_per_target)"
        return out.sort_values("target").reset_index(drop=True)

    # 마지막) zip/dir 전체 csv 중 target + r2컬럼 있는 걸 찾아서 처리(heuristic)
    # - zip이면 zip 내부 csv 전부 훑음, dir이면 rglob("*.csv")
    def _iter_csv_paths(run_path: str):
        if _is_zip(run_path):
            with zipfile.ZipFile(run_path, "r") as z:
                for n in z.namelist():
                    if n.lower().endswith(".csv"):
                        yield ("zip", n)
        else:
            for fp in Path(run_path).rglob("*.csv"):
                yield ("dir", fp)

    r2_candidates = ["test_r2", "PCR_R2", "r2", "R2", "test_R2", "Test_R2"]
    for kind, pth in _iter_csv_paths(run_path):
        try:
            if kind == "zip":
                df = _read_csv_zip(run_path, pth)
                where = f"ZIP::{pth}"
            else:
                df = _read_csv_dir(pth)
                where = f"DIR::{pth.as_posix()}"
        except Exception:
            continue

        if "target" not in df.columns:
            continue

        # 1) 단일 r2컬럼이 있으면 그걸 사용
        r2_col = None
        for c in r2_candidates:
            if c in df.columns:
                r2_col = c
                break

        if r2_col is not None:
            tmp = df.copy()
            tmp[r2_col] = pd.to_numeric(tmp[r2_col], errors="coerce")
            tmp = tmp.dropna(subset=["target", r2_col])
            idx = tmp.groupby("target")[r2_col].idxmax()
            best = tmp.loc[idx, ["target", r2_col]].copy()
            best = best.rename(columns={r2_col: f"{label}_best_test_r2"})
            best[f"{label}_source"] = where + f" (heuristic: groupby max {r2_col})"
            best[f"{label}_best_arg"] = np.nan
            return best.sort_values("target").reset_index(drop=True)

        # 2) wide format: 여러 R2 열이 있으면, 행별 max를 선택(모델 열 이름도 같이)
        r2_cols = [c for c in df.columns if re.search(r"(?i)r2", str(c))]
        if r2_cols:
            tmp = df.copy()
            for c in r2_cols:
                tmp[c] = pd.to_numeric(tmp[c], errors="coerce")
            tmp = tmp.dropna(subset=["target"])
            # 각 row에서 max R2
            tmp["row_best_r2"] = tmp[r2_cols].max(axis=1, skipna=True)
            tmp["row_best_model"] = tmp[r2_cols].idxmax(axis=1)
            tmp = tmp.dropna(subset=["row_best_r2"])
            # target별 max
            idx = tmp.groupby("target")["row_best_r2"].idxmax()
            best = tmp.loc[idx, ["target", "row_best_r2", "row_best_model"]].copy()
            best = best.rename(columns={
                "row_best_r2": f"{label}_best_test_r2",
                "row_best_model": f"{label}_best_arg",
            })
            best[f"{label}_source"] = where + " (heuristic: wide R2 cols row-max then target-max)"
            return best.sort_values("target").reset_index(drop=True)

    raise FileNotFoundError(
        f"[{label}] run_path에서 target별 test R2를 추출할 만한 CSV를 찾지 못했습니다: {run_path}"
    )


# =========================
# 3) 두 결과를 비교해서 CSV 저장
# =========================
paper_best = extract_best_test_r2(PAPER_RESULT, "paper")
exp_best   = extract_best_test_r2(EXP_RESULT, "exp")

compare = paper_best.merge(exp_best, on="target", how="outer")

# 차이 계산
compare["delta_test_r2(exp-paper)"] = compare["exp_best_test_r2"] - compare["paper_best_test_r2"]

# 보기 좋게 정렬: 개선 큰 순
compare_sorted = compare.sort_values("delta_test_r2(exp-paper)", ascending=False)

# 저장
compare_sorted.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

print("✅ saved:", OUT_CSV)
print("\n[Sanity check] sources used:")
print(" - paper:", paper_best["paper_source"].iloc[0] if "paper_source" in paper_best.columns else "n/a")
print(" - exp  :", exp_best["exp_source"].iloc[0] if "exp_source" in exp_best.columns else "n/a")

display(compare_sorted.head(20))

## pandas

In [ ]:
top_improve = compare_sorted.head(10)[["target","paper_best_test_r2","exp_best_test_r2","delta_test_r2(exp-paper)"]]
top_worse   = compare_sorted.tail(10)[["target","paper_best_test_r2","exp_best_test_r2","delta_test_r2(exp-paper)"]]

display(top_improve)
display(top_worse)

## test R^2 분포 비교

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.hist(compare["paper_best_test_r2"].dropna(), bins=20, alpha=0.6, label="paper best test R2")
plt.hist(compare["exp_best_test_r2"].dropna(), bins=20, alpha=0.6, label="exp best test R2")
plt.title("Best test R2 distribution (per target)")
plt.xlabel("best test R2")
plt.ylabel("count")
plt.grid(True)
plt.legend()
plt.show()

## 어떤 관능평가에서 차이가 큰지 산점도로 확인

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(compare["paper_best_test_r2"], compare["exp_best_test_r2"])
plt.plot([-10, 1], [-10, 1], linestyle="--")  # y=x 기준선
plt.xlim(-10, 1)
plt.ylim(-10, 1)
plt.title("paper vs exp (best test R2 per target)")
plt.xlabel("paper best test R2")
plt.ylabel("exp best test R2")
plt.grid(True)
plt.show()